# Dubbing 启动 Notebook
保存一份私有副本到 Drive，选择 GPU 后运行全部单元。每次启动获取 GitHub 最新工作 Notebook 和匹配源码。任务期间不会更新。
固定连接密码在下方配置一次，不使用 Colab Secret；MiniMax API 密钥仍使用 MINIMAX_API_KEY Secret。此文件包含密码时请保持私有，不提交 GitHub。

YouTube 登录 Cookie 可选：在 Colab Secrets 添加 YOUTUBE_COOKIES，填写完整 Netscape cookies.txt 内容，并开启 Notebook access。内容不会写入本文件或回传本机。


In [ ]:
# 改密码时同时在本机 GUI 的 Colab 连接中保存相同值。
DUBBING_PASSWORD = ''
DUBBING_VERSION = 'main'  # 也可填完整提交 SHA，以回退到已知版本。

# Optional: paste the complete Netscape cookies.txt into the Colab Secret.
from google.colab import userdata
try:
    YOUTUBE_COOKIES = userdata.get('YOUTUBE_COOKIES')
except userdata.SecretNotFoundError:
    YOUTUBE_COOKIES = ''
except userdata.NotebookAccessError as exc:
    raise RuntimeError('请允许此 Notebook 访问 YOUTUBE_COOKIES Secret') from None


In [ ]:
import json, re, subprocess, urllib.request
from getpass import getpass

if not DUBBING_PASSWORD:
    DUBBING_PASSWORD = getpass('本机固定 8 位连接密码: ').strip()
if not re.fullmatch(r'[0-9]{8}', DUBBING_PASSWORD):
    raise ValueError('密码必须为 8 位数字')
repository = 'https://github.com/Kaitzz/free-dubbing.git'
# Resolve once: the working notebook and application source use the SAME commit.
if not re.fullmatch(r'[0-9a-f]{40}', DUBBING_VERSION):
    if DUBBING_VERSION != 'main':
        raise ValueError('版本应为 main 或完整的提交 SHA')
    result = subprocess.check_output(['git', 'ls-remote', repository, 'refs/heads/main'], text=True)
    DUBBING_SOURCE_COMMIT = result.split()[0]
else:
    DUBBING_SOURCE_COMMIT = DUBBING_VERSION
url = f'https://raw.githubusercontent.com/Kaitzz/free-dubbing/{DUBBING_SOURCE_COMMIT}/notebooks/YouDub_GUI_Colab.ipynb'
with urllib.request.urlopen(url, timeout=60) as response:
    working = json.load(response)
print(f'Dubbing source version: {DUBBING_SOURCE_COMMIT}')
for index, cell in enumerate(working['cells'], 1):
    if cell['cell_type'] != 'code':
        continue
    print(f'--- Working notebook cell {index} ---', flush=True)
    try:
        exec(compile(''.join(cell['source']), f'<working-cell-{index}>', 'exec'), globals())
    except Exception as exc:
        raise RuntimeError(f'工作单元 {index} 失败；版本 {DUBBING_SOURCE_COMMIT[:7]}。查看上方原始错误。') from exc
